# TF-C Pretraining & Evaluation Runner (Kaggle GPU Ready)

### Quick Start Instructions:
1. **Enable Kaggle GPU**: In the Kaggle notebook sidebar, set **Accelerator** to **GPU P100** or **GPU T4 x2**.
2. **Enable Internet**: In the sidebar settings, toggle **Internet** to **On** (required for git clone & dataset download).
3. **Set Repository URL**: In Cell 2 below, paste your GitHub repository URL into `GITHUB_REPO`.
4. **Press Run All**: Click **Run All** (or `Shift + Enter` cell by cell).

---

In [ ]:
# ====================================================
# 1. USER CONFIGURATION (PASTE YOUR REPO & BRANCH HERE)
# ====================================================
GITHUB_REPO = "https://github.com/YoussefMoRabie/TFC-pretraining.git"
GITHUB_BRANCH = "main"

# ====================================================
# 2. TRAINING CONTROL FLAGS
# ====================================================
RUN_BENCHMARK = True           # Run 200-batch GPU speed/throughput benchmark
RUN_SCHEDULER_ABLATION = False # Run 5-epoch Constant vs Cosine LR ablation study
RUN_5_EPOCH_TEST = False       # Run single 5-epoch corrected pretraining + frozen KNN eval
RUN_40_EPOCH_FULL = False      # Run full 40-epoch pretraining + frozen KNN eval

import os
OUTPUT_DIR = "/kaggle/working/tfc_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Global configuration initialized. Output directory: {OUTPUT_DIR}")


In [ ]:
# ====================================================
# SECTION 1: REPOSITORY SETUP
# ====================================================
import os
import sys
import subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

print(f"Environment detected: IS_KAGGLE={IS_KAGGLE}")
print(f"Working directory: {WORK_DIR}")

if IS_KAGGLE:
    if GITHUB_REPO != "PASTE_MY_REPO_URL_HERE":
        repo_name = GITHUB_REPO.split("/")[-1].replace(".git", "")
        target_repo_dir = WORK_DIR / repo_name
        if not target_repo_dir.exists():
            print(f"Cloning repository {GITHUB_REPO} (branch: {GITHUB_BRANCH})...")
            subprocess.run(["git", "clone", "-b", GITHUB_BRANCH, GITHUB_REPO, str(target_repo_dir)], check=True)
        os.chdir(target_repo_dir)
        print(f"Changed directory to: {os.getcwd()}")
    else:
        print("Note: GITHUB_REPO is unset. Assuming code is already in current working directory.")

code_dir = os.path.abspath("code")
tfc_dir = os.path.abspath("code/TFC")
if code_dir not in sys.path:
    sys.path.append(code_dir)
if tfc_dir not in sys.path:
    sys.path.append(tfc_dir)

print(f"Current Working Directory: {os.getcwd()}")


In [ ]:
# ====================================================
# SECTION 2: ENVIRONMENT DIAGNOSTIC
# ====================================================
import sys
import torch

print("====================================================")
print(" ENVIRONMENT DIAGNOSTICS")
print("====================================================")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Current Device Index: {torch.cuda.current_device()}")
else:
    print("WARNING: CUDA is NOT available! Enable GPU in Kaggle settings (Accelerator -> GPU).")

print(f"Current Working Directory: {os.getcwd()}")


In [ ]:
# ====================================================
# SECTION 3: DATASET VERIFICATION & DOWNLOAD
# ====================================================
import os
import sys
import subprocess
import torch
from pathlib import Path

def verify_and_download_datasets():
    print("====================================================")
    print(" DATASET VERIFICATION & DOWNLOAD")
    print("====================================================")
    
    datasets_dir = Path("datasets")
    sleepeeg_dir = datasets_dir / "SleepEEG"
    epilepsy_dir = datasets_dir / "Epilepsy"
    
    sleepeeg_train = sleepeeg_dir / "train.pt"
    epilepsy_train = epilepsy_dir / "train.pt"
    
    if not (sleepeeg_train.exists() and epilepsy_train.exists()):
        print("Downloading missing datasets via download_datasets.py...")
        if os.path.exists("download_datasets.py"):
            subprocess.run([sys.executable, "download_datasets.py"], check=True)
        else:
            raise FileNotFoundError("download_datasets.py not found in working directory!")
    else:
        print("Datasets already present on disk. Skipping re-download.")
        
    required_files = {
        "SleepEEG Train": sleepeeg_dir / "train.pt",
        "SleepEEG Val": sleepeeg_dir / "val.pt",
        "SleepEEG Test": sleepeeg_dir / "test.pt",
        "Epilepsy Train": epilepsy_dir / "train.pt",
        "Epilepsy Val": epilepsy_dir / "val.pt",
        "Epilepsy Test": epilepsy_dir / "test.pt",
    }
    
    print("\nDataset Verification Summary:")
    for name, path in required_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing required dataset file: {path}")
        file_size_mb = path.stat().st_size / (1024 * 1024)
        data = torch.load(path)
        samples_shape = data["samples"].shape
        labels_shape = data["labels"].shape
        print(f"  [OK] {name:<15} | Size: {file_size_mb:>7.2f} MB | Samples: {str(samples_shape):<20} | Labels: {str(labels_shape)}")

verify_and_download_datasets()


In [ ]:
# ====================================================
# SECTION 4: GPU COMPATIBILITY & SMOKE TEST
# ====================================================
import torch
import torch.nn as nn
from model import TFC
from config_files.SleepEEG_Configs import Config as SleepEEGConfig

def run_gpu_smoke_test():
    print("====================================================")
    print(" GPU COMPATIBILITY & FORWARD-PASS SMOKE TEST")
    print("====================================================")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")
    
    configs = SleepEEGConfig()
    model = TFC(configs).to(device)
    
    batch_size = 2
    seq_len = int(configs.TSlength_aligned)
    x = torch.randn(batch_size, 1, seq_len).to(device)
    x_f = torch.fft.fft(x).abs().to(device)
    
    print(f"Input Time Domain Tensor Shape: {x.shape} (device: {x.device})")
    print(f"Input Freq Domain Tensor Shape: {x_f.shape} (device: {x_f.device})")
    
    model.eval()
    with torch.no_grad():
        h_t, z_t, h_f, z_f = model(x, x_f)
        
    print(f"Output Embedding h_t Shape: {h_t.shape}")
    print(f"Output Projection z_t Shape: {z_t.shape}")
    
    if torch.cuda.is_available():
        mem_allocated_mb = torch.cuda.memory_allocated() / (1024 * 1024)
        print(f"CUDA Memory Allocated: {mem_allocated_mb:.2f} MB")
        
    print("Smoke Test PASSED cleanly! Model operations are fully GPU-compatible.")

run_gpu_smoke_test()


In [ ]:
# ====================================================
# SECTION 5: TRAINING IMPLEMENTATION AUDIT
# ====================================================
import inspect

def audit_trainer_code():
    print("====================================================")
    print(" TRAINING IMPLEMENTATION AUDIT")
    print("====================================================")
    
    import trainer
    source_code = inspect.getsource(trainer.model_pretrain)
    
    print("Inspecting model_pretrain() source code from code/TFC/trainer.py:\n")
    print("----------------------------------------------------")
    lines = source_code.split("\n")
    for line in lines[:25]:
        print(line)
    print("----------------------------------------------------")
    
    in_loop = False
    for line in lines:
        if "for batch_idx" in line or "for " in line:
            in_loop = True
        if "zero_grad()" in line and in_loop:
            print("\n[VERIFIED] model_optimizer.zero_grad() is correctly located INSIDE the mini-batch loop!")
            return
            
    print("\n[WARNING] zero_grad() was NOT detected inside the loop!")

audit_trainer_code()


In [ ]:
# ====================================================
# SECTION 6: QUICK GPU THROUGHPUT BENCHMARK (200 BATCHES)
# ====================================================
import time
import json
import torch
import torch.nn as nn
from model import TFC
from loss import NTXentLoss_poly
from dataloader import data_generator
from config_files.SleepEEG_Configs import Config as SleepEEGConfig

def run_gpu_benchmark(num_batches=200):
    print("====================================================")
    print(f" QUICK GPU THROUGHPUT BENCHMARK ({num_batches} BATCHES)")
    print("====================================================")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    configs = SleepEEGConfig()
    
    sourcedata_path = "datasets/SleepEEG"
    targetdata_path = "datasets/Epilepsy"
    
    print("Loading Full SleepEEG dataset...")
    train_dl, _, _ = data_generator(sourcedata_path, targetdata_path, configs, "pre_train", subset=False)
    batches_per_epoch = len(train_dl)
    print(f"Batches per Epoch: {batches_per_epoch}")
    
    model = TFC(configs).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=configs.lr,
        betas=(configs.beta1, configs.beta2),
        weight_decay=3e-4
    )
    
    model.train()
    
    # Warmup batch
    for batch in train_dl:
        data, labels, aug1, data_f, aug1_f = batch
        data, aug1 = data.float().to(device), aug1.float().to(device)
        data_f, aug1_f = data_f.float().to(device), aug1_f.float().to(device)
        optimizer.zero_grad()
        h_t, z_t, h_f, z_f = model(data, data_f)
        h_t_aug, z_t_aug, h_f_aug, z_f_aug = model(aug1, aug1_f)
        nt_xent_criterion = NTXentLoss_poly(device, configs.batch_size, configs.Context_Cont.temperature, configs.Context_Cont.use_cosine_similarity)
        loss = 0.2 * (nt_xent_criterion(h_t, h_t_aug) + nt_xent_criterion(h_f, h_f_aug)) + nt_xent_criterion(z_t, z_f)
        loss.backward()
        optimizer.step()
        break
        
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        
    start_t = time.time()
    count = 0
    
    for batch_idx, (data, labels, aug1, data_f, aug1_f) in enumerate(train_dl):
        if count >= num_batches:
            break
        optimizer.zero_grad()
        data, aug1 = data.float().to(device), aug1.float().to(device)
        data_f, aug1_f = data_f.float().to(device), aug1_f.float().to(device)
        
        h_t, z_t, h_f, z_f = model(data, data_f)
        h_t_aug, z_t_aug, h_f_aug, z_f_aug = model(aug1, aug1_f)
        
        nt_xent_criterion = NTXentLoss_poly(device, configs.batch_size, configs.Context_Cont.temperature, configs.Context_Cont.use_cosine_similarity)
        loss = 0.2 * (nt_xent_criterion(h_t, h_t_aug) + nt_xent_criterion(h_f, h_f_aug)) + nt_xent_criterion(z_t, z_f)
        loss.backward()
        optimizer.step()
        count += 1
        
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    elapsed = time.time() - start_t
    sec_per_batch = elapsed / num_batches
    est_min_per_epoch = (sec_per_batch * batches_per_epoch) / 60.0
    est_min_5ep = est_min_per_epoch * 5
    est_hours_40ep = (est_min_per_epoch * 40) / 60.0
    
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0
    
    res_dict = {
        "sec_per_batch": sec_per_batch,
        "batches_per_epoch": batches_per_epoch,
        "est_min_per_epoch": est_min_per_epoch,
        "est_min_5ep": est_min_5ep,
        "est_hours_40ep": est_hours_40ep,
        "peak_gpu_mem_mb": peak_mem_mb
    }
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    bench_out_file = os.path.join(OUTPUT_DIR, "benchmark_results.json")
    with open(bench_out_file, "w") as f:
        json.dump(res_dict, f, indent=2)
    
    print("\n----------------------------------------------------")
    print(" BENCHMARK RESULTS")
    print("----------------------------------------------------")
    print(f"Throughput:              {sec_per_batch:.4f} sec/batch ({1/sec_per_batch:.2f} batch/sec)")
    print(f"Batches Per Epoch:       {batches_per_epoch}")
    print(f"Estimated Epoch Runtime: {est_min_per_epoch:.2f} minutes")
    print(f"Estimated 5-Epoch Run:   {est_min_5ep:.2f} minutes")
    print(f"Estimated 40-Epoch Run:  {est_hours_40ep:.2f} hours")
    print(f"Peak GPU Memory:         {peak_mem_mb:.2f} MB")
    print(f"Saved summary to:        {bench_out_file}")
    print("----------------------------------------------------")

if RUN_BENCHMARK:
    run_gpu_benchmark(200)
else:
    print("RUN_BENCHMARK is False. Skipping benchmark.")


In [ ]:
# ====================================================
# SECTION 7: PRETRAINING EXPERIMENT ENGINE
# ====================================================
import json
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

def eval_frozen_knn_epoch(model, device):
    data_path = "datasets/Epilepsy"
    train_dataset = torch.load(os.path.join(data_path, "train.pt"))
    test_dataset = torch.load(os.path.join(data_path, "test.pt"))
    
    train_x = train_dataset["samples"].float()
    train_y = train_dataset["labels"].long()
    test_x = test_dataset["samples"].float()
    test_y = test_dataset["labels"].long()
    
    train_x_f = torch.fft.fft(train_x).abs()
    test_x_f = torch.fft.fft(test_x).abs()
    
    from torch.utils.data import DataLoader, TensorDataset
    train_loader = DataLoader(dataset=TensorDataset(train_x, train_y, train_x_f), batch_size=128, shuffle=False)
    test_loader = DataLoader(dataset=TensorDataset(test_x, test_y, test_x_f), batch_size=128, shuffle=False)
    
    def extract_features(m, loader):
        m.eval()
        feats, lbls = [], []
        with torch.no_grad():
            for x_t, y, x_f in loader:
                x_t, x_f = x_t.to(device), x_f.to(device)
                _, z_t, _, z_f = m(x_t, x_f)
                fea = torch.cat((z_t, z_f), dim=1)
                feats.append(fea.cpu().numpy())
                lbls.append(y.numpy())
        return np.concatenate(feats, axis=0), np.concatenate(lbls, axis=0)

    train_feat, train_lbl = extract_features(model, train_loader)
    test_feat, test_lbl = extract_features(model, test_loader)
    
    knn = KNeighborsClassifier(n_neighbors=5, weights="uniform", metric="minkowski", p=2)
    knn.fit(train_feat, train_lbl)
    
    preds = knn.predict(test_feat)
    probs = knn.predict_proba(test_feat)[:, 1]
    
    return {
        "acc": float(accuracy_score(test_lbl, preds) * 100.0),
        "f1": float(f1_score(test_lbl, preds, average="macro") * 100.0),
        "precision": float(precision_score(test_lbl, preds, average="macro") * 100.0),
        "recall": float(recall_score(test_lbl, preds, average="macro") * 100.0),
        "auroc": float(roc_auc_score(test_lbl, probs) * 100.0),
        "auprc": float(average_precision_score(test_lbl, probs) * 100.0)
    }

def run_pretraining_experiment(num_epochs, exp_label):
    print("====================================================")
    print(f" STARTING PRETRAINING EXPERIMENT: {exp_label} ({num_epochs} EPOCHS)")
    print("====================================================")
    
    exp_output_dir = os.path.join(OUTPUT_DIR, exp_label)
    os.makedirs(exp_output_dir, exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    configs = SleepEEGConfig()
    configs.num_epoch = num_epochs
    
    seed = 42
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    sourcedata_path = "datasets/SleepEEG"
    targetdata_path = "datasets/Epilepsy"
    
    train_dl, _, _ = data_generator(sourcedata_path, targetdata_path, configs, "pre_train", subset=False)
    
    model = TFC(configs).to(device)
    init_params = {name: p.data.clone() for name, p in model.named_parameters()}
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=configs.lr,
        betas=(configs.beta1, configs.beta2),
        weight_decay=3e-4
    )
    
    records = []
    
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_losses = []
        grad_norms = []
        epoch_start_t = time.time()
        
        for batch_idx, (data, labels, aug1, data_f, aug1_f) in enumerate(train_dl):
            optimizer.zero_grad()
            data, aug1 = data.float().to(device), aug1.float().to(device)
            data_f, aug1_f = data_f.float().to(device), aug1_f.float().to(device)
            
            h_t, z_t, h_f, z_f = model(data, data_f)
            h_t_aug, z_t_aug, h_f_aug, z_f_aug = model(aug1, aug1_f)
            
            nt_xent_criterion = NTXentLoss_poly(device, configs.batch_size, configs.Context_Cont.temperature, configs.Context_Cont.use_cosine_similarity)
            loss = 0.2 * (nt_xent_criterion(h_t, h_t_aug) + nt_xent_criterion(h_f, h_f_aug)) + nt_xent_criterion(z_t, z_f)
            loss.backward()
            
            g_sq = sum(p.grad.data.norm(2).item()**2 for p in model.parameters() if p.grad is not None)
            grad_norms.append(np.sqrt(g_sq))
            
            optimizer.step()
            epoch_losses.append(loss.item())
            
        dur = time.time() - epoch_start_t
        avg_loss = float(np.mean(epoch_losses))
        roll100 = float(np.mean(epoch_losses[-100:]))
        mean_g = float(np.mean(grad_norms))
        max_g = float(np.max(grad_norms))
        
        p_norm = float(np.sqrt(sum(p.data.norm(2).item()**2 for p in model.parameters() if p.requires_grad)))
        p_dist = float(np.sqrt(sum((p.data - init_params[n]).norm(2).item()**2 for n, p in model.named_parameters() if n in init_params)))
        
        knn_res = eval_frozen_knn_epoch(model, device)
        
        print(f"[Epoch {epoch:02d}/{num_epochs:02d}] ({dur:.1f}s) Loss: {avg_loss:.4f} | Roll100: {roll100:.4f} | GradNorm: {mean_g:.4f} | ParamDist: {p_dist:.2f}")
        print(f"   --> Frozen KNN: Acc={knn_res['acc']:.2f}% | F1={knn_res['f1']:.2f}% | AUROC={knn_res['auroc']:.2f}% | AUPRC={knn_res['auprc']:.2f}%")
        
        records.append({
            "epoch": epoch,
            "duration_sec": dur,
            "avg_loss": avg_loss,
            "roll100_loss": roll100,
            "mean_grad_norm": mean_g,
            "max_grad_norm": max_g,
            "param_norm": p_norm,
            "param_dist": p_dist,
            "knn_results": knn_res
        })
        
        ckp_path = os.path.join(exp_output_dir, f"ckp_epoch_{epoch}.pt")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, ckp_path)
        
    torch.save({'model_state_dict': model.state_dict()}, os.path.join(exp_output_dir, "ckp_last.pt"))
    with open(os.path.join(exp_output_dir, "results.json"), "w") as f:
        json.dump(records, f, indent=2)
    print(f"Saved experiment outputs to: {exp_output_dir}")

if RUN_5_EPOCH_TEST:
    run_pretraining_experiment(5, "5_epoch_test")

if RUN_40_EPOCH_FULL:
    run_pretraining_experiment(40, "40_epoch_full")


In [ ]:
# ====================================================
# SECTION 7B: 5-EPOCH SCHEDULER ABLATION STUDY
# ====================================================
import random
import json
import time
import numpy as np
import torch

def run_scheduler_ablation():
    print("====================================================")
    print(" STARTING 5-EPOCH SCHEDULER ABLATION STUDY")
    print("====================================================")
    
    ablation_dir = os.path.join(OUTPUT_DIR, "scheduler_ablation")
    os.makedirs(ablation_dir, exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    configs = SleepEEGConfig()
    configs.num_epoch = 5
    num_epochs = 5
    seed = 42
    
    sourcedata_path = "datasets/SleepEEG"
    targetdata_path = "datasets/Epilepsy"
    
    def reset_all_seeds(s=42):
        random.seed(s)
        np.random.seed(s)
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
            
    def run_single_condition(condition_name, use_scheduler):
        print(f"\n>>> Running Condition: {condition_name} (use_scheduler={use_scheduler})")
        cond_dir = os.path.join(ablation_dir, condition_name)
        os.makedirs(cond_dir, exist_ok=True)
        
        # RESET SEEDS BEFORE DATALOADER AND MODEL CREATION
        reset_all_seeds(seed)
        
        train_dl, _, _ = data_generator(sourcedata_path, targetdata_path, configs, "pre_train", subset=False)
        model = TFC(configs).to(device)
        init_params = {name: p.data.clone() for name, p in model.named_parameters()}
        
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=configs.lr,
            betas=(configs.beta1, configs.beta2),
            weight_decay=3e-4
        )
        
        scheduler = None
        if use_scheduler:
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=num_epochs, eta_min=0.0
            )
            
        records = []
        
        for epoch in range(1, num_epochs + 1):
            current_lr = optimizer.param_groups[0]['lr']
            model.train()
            epoch_losses = []
            grad_norms = []
            epoch_start_t = time.time()
            
            for batch_idx, (data, labels, aug1, data_f, aug1_f) in enumerate(train_dl):
                optimizer.zero_grad()
                data, aug1 = data.float().to(device), aug1.float().to(device)
                data_f, aug1_f = data_f.float().to(device), aug1_f.float().to(device)
                
                h_t, z_t, h_f, z_f = model(data, data_f)
                h_t_aug, z_t_aug, h_f_aug, z_f_aug = model(aug1, aug1_f)
                
                nt_xent_criterion = NTXentLoss_poly(device, configs.batch_size, configs.Context_Cont.temperature, configs.Context_Cont.use_cosine_similarity)
                loss = 0.2 * (nt_xent_criterion(h_t, h_t_aug) + nt_xent_criterion(h_f, h_f_aug)) + nt_xent_criterion(z_t, z_f)
                loss.backward()
                
                g_sq = sum(p.grad.data.norm(2).item()**2 for p in model.parameters() if p.grad is not None)
                grad_norms.append(np.sqrt(g_sq))
                
                optimizer.step()
                epoch_losses.append(loss.item())
                
            # STEP SCHEDULER AT END OF EPOCH
            if scheduler is not None:
                scheduler.step()
                
            dur = time.time() - epoch_start_t
            avg_loss = float(np.mean(epoch_losses))
            roll100 = float(np.mean(epoch_losses[-100:]))
            mean_g = float(np.mean(grad_norms))
            max_g = float(np.max(grad_norms))
            
            p_norm = float(np.sqrt(sum(p.data.norm(2).item()**2 for p in model.parameters() if p.requires_grad)))
            p_dist = float(np.sqrt(sum((p.data - init_params[n]).norm(2).item()**2 for n, p in model.named_parameters() if n in init_params)))
            
            knn_res = eval_frozen_knn_epoch(model, device)
            
            print(f"[{condition_name} | Epoch {epoch}/{num_epochs}] LR={current_lr:.6f} | ({dur:.1f}s) Loss: {avg_loss:.4f} | Roll100: {roll100:.4f} | GradNorm: {mean_g:.4f} | ParamDist: {p_dist:.2f}")
            print(f"   --> Frozen KNN: Acc={knn_res['acc']:.2f}% | F1={knn_res['f1']:.2f}% | AUROC={knn_res['auroc']:.2f}% | AUPRC={knn_res['auprc']:.2f}%")
            
            records.append({
                "epoch": epoch,
                "lr": current_lr,
                "duration_sec": dur,
                "avg_loss": avg_loss,
                "roll100_loss": roll100,
                "mean_grad_norm": mean_g,
                "max_grad_norm": max_g,
                "param_norm": p_norm,
                "param_dist": p_dist,
                "knn_results": knn_res
            })
            
            ckp_path = os.path.join(cond_dir, f"ckp_epoch_{epoch}.pt")
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'lr': current_lr}, ckp_path)
            
        with open(os.path.join(cond_dir, "results.json"), "w") as f:
            json.dump(records, f, indent=2)
            
        return records
        
    res_const = run_single_condition("constant_lr_5ep", use_scheduler=False)
    res_cosine = run_single_condition("cosine_lr_5ep", use_scheduler=True)
    
    # COMPARISON TABLE & SUMMARY
    print("\n====================================================")
    print(" SCHEDULER ABLATION COMPARISON SUMMARY")
    print("====================================================")
    print(f"{'Epoch':<6} | {'Constant F1':<12} | {'Cosine F1':<12} | {'Constant AUROC':<15} | {'Cosine AUROC':<15} | {'Constant Loss':<14} | {'Cosine Loss':<14} | {'Cosine LR':<10}")
    print("-" * 110)
    
    for i in range(num_epochs):
        ep = i + 1
        c_f1 = res_const[i]["knn_results"]["f1"]
        cos_f1 = res_cosine[i]["knn_results"]["f1"]
        c_auroc = res_const[i]["knn_results"]["auroc"]
        cos_auroc = res_cosine[i]["knn_results"]["auroc"]
        c_loss = res_const[i]["avg_loss"]
        cos_loss = res_cosine[i]["avg_loss"]
        cos_lr = res_cosine[i]["lr"]
        print(f"{ep:<6} | {c_f1:>10.2f}% | {cos_f1:>10.2f}% | {c_auroc:>13.2f}% | {cos_auroc:>13.2f}% | {c_loss:>14.4f} | {cos_loss:>14.4f} | {cos_lr:>10.6f}")
        
    # FIND BEST EPOCHS
    best_const_ep = max(range(num_epochs), key=lambda idx: res_const[idx]["knn_results"]["f1"])
    best_cos_ep = max(range(num_epochs), key=lambda idx: res_cosine[idx]["knn_results"]["f1"])
    
    best_c_f1 = res_const[best_const_ep]["knn_results"]["f1"]
    best_cos_f1 = res_cosine[best_cos_ep]["knn_results"]["f1"]
    
    prevents_degradation = res_cosine[-1]["knn_results"]["f1"] >= res_cosine[best_cos_ep]["knn_results"]["f1"] - 1.0
    improves_peak = best_cos_f1 > best_c_f1
    
    summary_dict = {
        "constant_results": res_const,
        "cosine_results": res_cosine,
        "best_constant_epoch": best_const_ep + 1,
        "best_constant_f1": best_c_f1,
        "best_cosine_epoch": best_cos_ep + 1,
        "best_cosine_f1": best_cos_f1,
        "prevents_degradation": prevents_degradation,
        "improves_peak": improves_peak
    }
    
    with open(os.path.join(ablation_dir, "ablation_summary.json"), "w") as f:
        json.dump(summary_dict, f, indent=2)
        
    print("\n--- ABLATION EVALUATION ANSWERS ---")
    print(f"1. Best Constant LR Epoch: Epoch {best_const_ep + 1} (F1: {best_c_f1:.2f}%)")
    print(f"2. Best Cosine LR Epoch:   Epoch {best_cos_ep + 1} (F1: {best_cos_f1:.2f}%)")
    print(f"3. Best Overall Frozen KNN F1: {max(best_c_f1, best_cos_f1):.2f}%")
    print(f"4. Does Cosine LR prevent Epoch-3-to-5 transfer degradation? {'YES' if prevents_degradation else 'NO'}")
    print(f"5. Does Cosine LR improve peak performance or only preserve performance? {'Improves Peak Performance' if improves_peak else 'Preserves Performance'}")
    print(f"6. Is there any justification for running 40 epochs? {'NO - Peak transfer reached in <= 5 epochs' if max(best_const_ep, best_cos_ep) <= 3 else 'YES - Performance continues improving'}")
    print(f"\nAblation output saved to: {ablation_dir}")

if RUN_SCHEDULER_ABLATION:
    run_scheduler_ablation()
else:
    print("RUN_SCHEDULER_ABLATION is False. Skipping scheduler ablation study.")


In [ ]:
# ====================================================
# SECTION 8: PACKAGING OUTPUTS FOR DOWNLOAD
# ====================================================
import os
import shutil

def export_outputs_zip():
    print("====================================================")
    print(" PACKAGING OUTPUTS FOR EXPORT")
    print("====================================================")
    
    zip_path = "/kaggle/working/tfc_outputs.zip"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if len(os.listdir(OUTPUT_DIR)) == 0:
        summary_file = os.path.join(OUTPUT_DIR, "run_summary.txt")
        with open(summary_file, "w") as f:
            f.write("TF-C Kaggle Execution initialized successfully.\n")
        print(f"Created placeholder summary file: {summary_file}")
        
    shutil.make_archive("/kaggle/working/tfc_outputs", 'zip', OUTPUT_DIR)
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"[SUCCESS] Zip archive created: {zip_path} ({size_mb:.2f} MB)")
    print("You can download tfc_outputs.zip directly from the Kaggle Output panel!")

export_outputs_zip()
